# 02 — Signal Analysis

Validate that the rolling-mean overnight return score has predictive power.

**Prerequisites**: `01_data_audit.ipynb` passed.

Key questions:
1. Do overnight returns autocorrelate? (Is there persistence?)
2. Do high-score stocks deliver higher next-session overnight returns? (IC / quintile test)
3. Is the effect stable over time and across weekdays?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from config import NIFTY50_SYMBOLS, LOOKBACK_SESSIONS
from utils import load_all_ohlc, compute_overnight_returns, compute_scores

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
ohlc_dict    = load_all_ohlc(NIFTY50_SYMBOLS)
overnight_df = compute_overnight_returns(ohlc_dict)
scores_df    = compute_scores(overnight_df, lookback=LOOKBACK_SESSIONS)

print(f'overnight_df: {overnight_df.shape}  ({overnight_df.index[0].date()} → {overnight_df.index[-1].date()})')
print(f'scores_df:    {scores_df.shape}  (first valid row: {scores_df.dropna(how="all").index[0].date()})')

## Cell 1 — Autocorrelation of overnight returns

Expect positive autocorrelation at lag 1–5 for most stocks. This is the theoretical basis of the strategy.

In [ ]:
lags = [1, 5, 10, 20]
acf_data = {}

for sym in overnight_df.columns:
    col = overnight_df[sym].dropna()
    acf_data[sym] = {lag: col.autocorr(lag=lag) for lag in lags}

acf_df = pd.DataFrame(acf_data).T
acf_df.columns = [f'lag_{l}' for l in lags]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    acf_df,
    cmap='RdYlGn',
    center=0,
    vmin=-0.15,
    vmax=0.15,
    annot=True,
    fmt='.2f',
    linewidths=0.3,
    ax=ax,
)
ax.set_title(f'Autocorrelation of Overnight Returns by Lag (lookback={LOOKBACK_SESSIONS})')
plt.tight_layout()
plt.show()

print('Mean autocorrelation across stocks:')
for lag in lags:
    col = f'lag_{lag}'
    pct_pos = (acf_df[col] > 0).mean()
    print(f'  lag {lag:>2}: mean={acf_df[col].mean():.4f}  pct_positive={pct_pos:.0%}')

## Cell 2 — Quintile backtest

Each session: sort stocks into 5 quintiles by score. Compute next-session overnight return per quintile.
Expect monotonic increase Q1 → Q5.

In [ ]:
# Align: score at t predicts overnight return at t+1
score_t  = scores_df.iloc[:-1]   # all rows except last
return_t1 = overnight_df.iloc[1:] # all rows except first, shifted forward
return_t1.index = score_t.index   # align index

quintile_returns = {q: [] for q in range(1, 6)}

for dt in score_t.index:
    row_s = score_t.loc[dt].dropna()
    row_r = return_t1.loc[dt].dropna()
    common = row_s.index.intersection(row_r.index)
    if len(common) < 10:
        continue
    row_s = row_s[common]
    row_r = row_r[common]
    quintiles = pd.qcut(row_s, q=5, labels=False) + 1
    for q in range(1, 6):
        mask = quintiles == q
        quintile_returns[q].append(row_r[mask].mean())

q_means = {q: np.nanmean(v) * 100 for q, v in quintile_returns.items()}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(list(q_means.keys()), list(q_means.values()),
              color=['#d73027','#fc8d59','#fee090','#91cf60','#1a9850'], edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Score Quintile (1=lowest, 5=highest)')
ax.set_ylabel('Mean Next-Session Overnight Return (%)')
ax.set_title('Quintile Returns: Does High Score → High Next Return?')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=3))
for bar, val in zip(bars, q_means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print('Quintile mean next-session overnight returns:')
for q, v in q_means.items():
    print(f'  Q{q}: {v:.5f}%')

q1_to_q5_spread = q_means[5] - q_means[1]
print(f'\nQ5 - Q1 spread: {q1_to_q5_spread:.5f}%')
print('→ Positive spread with monotonic progression = signal works.')

## Cell 3 — Information Coefficient (IC)

Spearman rank correlation between score and next-day overnight return per session.
Expect mean IC > 0.03.

In [ ]:
ic_series = []

for dt in score_t.index:
    row_s = score_t.loc[dt].dropna()
    row_r = return_t1.loc[dt].dropna()
    common = row_s.index.intersection(row_r.index)
    if len(common) < 10:
        continue
    ic, _ = stats.spearmanr(row_s[common], row_r[common])
    ic_series.append((dt, ic))

ic_df = pd.DataFrame(ic_series, columns=['date','ic']).set_index('date')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily IC
axes[0].plot(ic_df.index, ic_df['ic'], alpha=0.4, linewidth=0.7, color='steelblue', label='Daily IC')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(ic_df['ic'].mean(), color='green', linewidth=1.2, linestyle='--', label=f'Mean IC={ic_df["ic"].mean():.4f}')
axes[0].set_title('Daily IC (Spearman: score vs next overnight return)')
axes[0].legend()

# Rolling 60-session mean IC
rolling_ic = ic_df['ic'].rolling(60, min_periods=20).mean()
axes[1].plot(rolling_ic.index, rolling_ic, linewidth=1.2, color='darkblue', label='60-session rolling mean IC')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].fill_between(rolling_ic.index, 0, rolling_ic, where=(rolling_ic > 0), alpha=0.2, color='green')
axes[1].fill_between(rolling_ic.index, 0, rolling_ic, where=(rolling_ic < 0), alpha=0.2, color='red')
axes[1].set_title('60-Session Rolling Mean IC')
axes[1].legend()

plt.tight_layout()
plt.show()

mean_ic = ic_df['ic'].mean()
pct_pos_ic = (ic_df['ic'] > 0).mean()
print(f'Mean IC:          {mean_ic:.4f}')
print(f'IC > 0 on:        {pct_pos_ic:.1%} of sessions')
if mean_ic >= 0.05:
    print('✓ Strong signal (IC ≥ 0.05)')
elif mean_ic >= 0.03:
    print('✓ Acceptable signal (IC 0.03–0.05). Expect modest edge after costs.')
else:
    print('⚠ Weak signal (IC < 0.03). May not survive transaction costs.')

## Cell 4 — Day-of-week effect

In [ ]:
day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri'}

# Top-10 overnight returns by weekday
top10_rets_by_day = {d: [] for d in range(5)}

for dt in score_t.index:
    dow = dt.weekday()
    if dow not in top10_rets_by_day:
        continue
    row_s = score_t.loc[dt].dropna()
    row_r = return_t1.loc[dt].dropna()
    common = row_s.index.intersection(row_r.index)
    if len(common) < 10:
        continue
    top10 = row_s[common].nlargest(10).index
    top10_rets_by_day[dow].append(row_r[common][top10].mean())

dow_means = {day_names[d]: np.nanmean(v) * 100 for d, v in top10_rets_by_day.items()}
dow_counts = {day_names[d]: len(v) for d, v in top10_rets_by_day.items()}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(dow_means.keys()), list(dow_means.values()), color='steelblue', edgecolor='black', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Top-10 Overnight Return (%)')
ax.set_title('Day-of-Week Effect: Mean Overnight Return of Top-10 Stocks')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=3))
plt.tight_layout()
plt.show()

for day, mean_r in dow_means.items():
    print(f'  {day}: {mean_r:.5f}%  (n={dow_counts[day]} sessions)')